In [1]:
import os
import sys
sys.path.insert(0, '')
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import torch
import numpy as np
from PIL import Image
from model import AdaptModel, data_transform, inverse_data_transform
from torchvision import transforms
from matplotlib import pyplot as plt
from tqdm import tqdm
import gc
import pdb
from thop import profile

rank = 'cuda:0'
model = AdaptModel(rank=rank, num_class=9, pretrained='./pretrained/MKF/model.pth')
model.ae.encoder = model.ae.encoder.to(rank)
model.ae.decoder = model.ae.decoder.to(rank)
model.IKRModule = model.IKRModule.to(rank)
model.CKGModule = model.CKGModule.to(rank)
model.FusionModule = model.FusionModule.to(rank)
model.FusionModule.eval()
print('done')

[Info]: Loading config from [./pretrained/IKR/config.yaml] as yaml file.
[Info]: Loading config from [./pretrained/CKG/config.yaml] as yaml file.
==> load pretrained model
done


In [ ]:
img = torch.ones(1, 3, 480, 640).to(rank)
raw_z = torch.randn(1, 8, 120, 160).to(rank)
ikr_xt = ckg_xt = fuse_xt = torch.randn_like(raw_z, dtype=torch.float32).to(rank) 
batch, c, h, w = raw_z.shape
seg = torch.randn(batch, 9, h, w).to(rank) 

with torch.no_grad():
        t = (torch.ones(batch)).to(rank)
        next_t = (torch.ones(batch)).to(rank)
        at = model.alphas_cumprod_pre.index_select(0, t.long()+1).view(-1, 1, 1, 1)
        at_next = model.alphas_cumprod_pre.index_select(0, next_t.long()+1).view(-1, 1, 1, 1)
        flops1, param1 = profile(model.ae.encoder, inputs=(img,), verbose=False)
        flops2, param2 = profile(model.IKRModule, inputs=(torch.cat([raw_z, ikr_xt], dim=1), t), verbose=False)
        flops3, param3 = profile(model.CKGModule, inputs=(torch.cat([raw_z, ckg_xt], dim=1), t), verbose=False)
        fusion_input = torch.cat([ikr_xt, ckg_xt, fuse_xt], dim=1)
        flops4, param4 = profile(model.FusionModule, inputs=(fusion_input, ikr_xt, ckg_xt, t, seg), verbose=False)
        flops5, param5 = profile(model.ae.decoder, inputs=(raw_z,), verbose=False)

print(f"{flops1:.2e}, {flops2:.2e}, {flops3:.2e}, {flops4:.2e}, {flops5:.2e}, " 
      f"{flops1+flops2+flops3+flops4+flops5:.2e}\n")
print(f"{param1:.2e}, {param2:.2e}, {param3:.2e}, {param4:.2e}, {param5:.2e}, "
      f"{param1+param2+param3+param4+param5:.2e}")